# End-to-End QPU Workflow on LQCloud

This tutorial runs a variational algorithm on a superconducting quantum processor of [LQCloud](https://cloud.logicalqubit.com) (逻辑比特), going through the steps of a real hardware experiment with TensorCircuit:

1. read the device calibration;
2. compile circuits locally into native gates on the physical qubits with the lowest errors (noise-aware compilation);
3. submit tasks, with every submission recorded in a local task store;
4. evaluate Pauli string expectations with grouped measurements and readout error mitigation by `batch_expectation_ps`;
5. compile a parameterized ansatz once and run parameter-shift gradient descent, each step being one batch submission.

The same code runs on the other providers of `tc.cloud`, only the device name changes. With `device=None`, `batch_expectation_ps` gives the exact values by simulation, so the whole workflow can be developed offline and switched to hardware at the end.

**Requirements:** the `lqcloud` SDK (`pip install "lqcloud>=0.5.1,<0.6"`, Python 3.11 or 3.12), Qiskit (version 2 or later is recommended for the noise-aware layout), and an LQCloud API key. Querying devices and compiling are free, while the cells submitting tasks are charged by LQCloud, so the notebook is shown without outputs.

In [ ]:
import numpy as np
import sympy

import tensorcircuit as tc
from tensorcircuit.cloud import apis
from tensorcircuit.cloud.wrapper import batch_expectation_ps
from tensorcircuit.compiler import DefaultCompiler

## Token and task store

The API key is read from the `TC_TOKEN_LQCLOUD` environment variable, or set in Python as below. It is kept in the current process only, unless `cached=True` also saves it on the local computer.

```python
apis.set_token("<your LQCloud API key>", provider="lqcloud")
```

We also turn on the local task store. Every following submission, including those inside `batch_expectation_ps`, is recorded in `~/.tensorcircuit/tasks`: the circuit, the compiled program, the request actually sent, the calibration of the qubits used, and the results once they are queried. A record is written **before** the request is sent, so that no paid task is lost even if the connection drops.

In [ ]:
store = apis.set_store(True)

## Devices and calibration

Querying devices and calibration data is free.

In [ ]:
apis.list_devices(provider="lqcloud")

We use `MQ02`, a 24-qubit chain. `list_properties` returns the device configuration, with the latest calibration standardized as `bits` (T1 and T2 in μs, readout and single qubit error rates) and `links` (the CZ error rate of each coupler).

In [ ]:
d = apis.get_device("lqcloud::MQ02")
p = d.list_properties()
print("calibrated at", p["calibration_time"])
print("native gates:", d.native_gates())
print("qubit 0:", p["bits"][0])
cz = sorted((link["CZErrRate"], q) for q, link in p["links"].items() if q[0] < q[1])
print("best couplers:", [(q, round(e, 4)) for e, q in cz[:3]])
print("worst couplers:", [(q, round(e, 4)) for e, q in cz[-3:]])

## Noise-aware compilation

LQCloud runs the submitted native gates as they are, so TensorCircuit compiles each circuit locally with Qiskit: it chooses the physical qubits, routes on the coupling map and decomposes into the native gates `rz`, `sx` (`xhalf`), `x` and `cz`. By default the layout is **noise aware**: the qubits and couplers are scored by their calibrated errors, see `tc.compiler.qiskit_compiler.properties_to_target`, and qubits or couplers of unknown errors are never used.

`compile_circuit` shows what would be sent for a 5-qubit GHZ circuit, without submitting anything. We compare the noise-aware layout with a layout chosen on the topology only.

In [ ]:
n = 5
ghz = tc.Circuit(n)
ghz.h(0)
for i in range(n - 1):
    ghz.cx(i, i + 1)


def summary(compilation):
    cal = compilation["calibration"]
    cz_err = sum(link["CZErrRate"] for link in cal["links"])
    ro_err = sum(b["ReadoutF0Err"] + b["ReadoutF1Err"] for b in cal["bits"]) / 2
    return compilation["physical_qubits"], round(cz_err, 4), round(ro_err, 4)


for noise_aware in [True, False]:
    _, compilation = tc.cloud.lqcloud.compile_circuit(ghz, p, noise_aware=noise_aware)
    print("noise aware" if noise_aware else "topology only", summary(compilation))

Each line gives the physical qubits, the summed CZ error of the couplers used and the summed average readout error. On a chain the layout only chooses where the GHZ chain sits, while on a 2D device it also avoids broken couplers. The calibration changes over time, so compile right before submitting.

## Submitting tasks

`submit_task` compiles the circuit as above and submits it. Without measurement instructions, all qubits are measured; the bitstrings of the results are indexed by the qubits of the circuit as written, not the physical ones.

In [ ]:
t = apis.submit_task(device=d, circuit=ghz, shots=1000)
counts = t.results()
dict(sorted(counts.items(), key=lambda kv: -kv[1])[:4])

The task keeps what was prepared and sent, which is also in the task store:

- `t.prepared["compilation"]`: the compiled native instructions on the physical qubits, the compiler options and the calibration of the qubits used;
- `t.prepared["payload"]`: the request body as actually sent by the SDK, e.g. with the readout pulses it adds;
- `t.details()["tensorcircuit"]["positional_logical_mapping"]`: from the bit of the results to the physical qubit, e.g. for readout error mitigation.

In [ ]:
details = t.details()
print("physical qubits:", t.prepared["compilation"]["physical_qubits"])
print("mapping:", details["tensorcircuit"]["positional_logical_mapping"])
print("archive:", details["tensorcircuit"]["archive"])
print("cost:", details["lqcloud"]["cost"], details["lqcloud"]["currency"])

## The task store

The store is a SQLite index with a compressed JSON record per circuit. It can be searched by provider, device, state, batch or time, and any record can be turned back into a `Task`, e.g. to fetch results in a new Python session.

In [ ]:
for row in store.find(provider="lqcloud", limit=3):
    print(row["remote_id"], row["device"], row["state"], row["submission_status"])

record = store.get(local_id=t.local_id)
print(sorted(record))

In [ ]:
t_again = store.task(t.local_id)
t_again.results() == counts

Paid submissions are **sent exactly once and never retried automatically**, since a retry after a lost response could run and charge the tasks twice. When a request fails:

- `tc.cloud.utils.SubmissionRejected` means the provider explicitly rejected it, so nothing was submitted and its record is removed;
- `tc.cloud.utils.SubmissionUnknown` means the request may have been accepted. Its records stay in the store with the `unknown` submission status. After finding the task on the LQCloud console, bind its id to recover it, instead of submitting again:

```python
for row in store.find(submission_status="unknown"):
    print(row["local_id"], row["created_at"])
t = store.bind("<local id>", "<task id on LQCloud>")
t.results()
```

`store.export("tasks.jsonl.gz", provider="lqcloud")` writes the selected records to one self-contained file, e.g. to archive the raw data of an experiment.

## Expectations with `batch_expectation_ps`

Consider the two-qubit transverse field Ising model

$$H = -Z_0Z_1 - X_0 - X_1,$$

with Pauli strings encoded as lists of 0, 1, 2, 3 for I, X, Y, Z. `batch_expectation_ps` evaluates the weighted sum on a device in one call: it compiles the circuit, groups qubit-wise commuting Pauli strings into measurement circuits, here $\{Z_0Z_1\}$ and $\{X_0, X_1\}$, submits them in one batch, and corrects readout errors by the calibration of the measured physical qubits. With `device=None`, the same call returns the exact value.

The ansatz is a `SymbolCircuit` with free parameters.

In [ ]:
pss = [[3, 3], [1, 0], [0, 1]]
ws = [-1.0, -1.0, -1.0]
thetas = sympy.symbols("theta0:4", real=True)


def ansatz():
    c = tc.SymbolCircuit(2)
    c.ry(0, theta=thetas[0])
    c.ry(1, theta=thetas[1])
    c.cz(0, 1)
    c.ry(0, theta=thetas[2])
    c.ry(1, theta=thetas[3])
    return c


ground = np.linalg.eigvalsh(tc.quantum.PauliStringSum2Dense(pss, ws))[0]
print("ground state energy:", np.real(ground))

theta0 = np.full(4, -0.5)
bind = lambda theta: dict(zip(thetas, theta))
print(
    "exact energy at theta0:",
    batch_expectation_ps(ansatz(), pss, ws=ws, params=bind(theta0)),
)

## Compile once, run many times

A variational algorithm evaluates the same circuit at many parameters. We compile the `SymbolCircuit` once with the noise-aware layout; the symbols are kept in the compiled gates. Passing the compiled circuit with its `info` to `batch_expectation_ps` skips compilation, so every evaluation runs on the same physical qubits, and `params` binds a list of parameter values whose circuits are all submitted in one batch.

In [ ]:
compiled, info = DefaultCompiler(properties=p)(ansatz())
print("physical qubits:", info["logical_physical_mapping"])
print("free symbols:", sorted(compiled.free_symbols(), key=str))

The parameter shift rule gives the exact gradient of each `ry` angle as

$$\partial_k E(\theta) = \frac{E(\theta + \tfrac{\pi}{2} e_k) - E(\theta - \tfrac{\pi}{2} e_k)}{2}.$$

One gradient descent step thus needs the energy at the current parameters and at 8 shifted ones: 9 parameter sets times 2 measurement groups, i.e. one batch of 18 circuits.

In [ ]:
shifts = np.pi / 2 * np.eye(4)


def energy_and_gradient(theta, device=None, shots=1000):
    values = [theta] + [theta + sign * s for s in shifts for sign in (1, -1)]
    es = batch_expectation_ps(
        compiled,
        pss,
        device=device,
        ws=ws,
        shots=shots,
        params=[bind(v) for v in values],
        info=info,
    )
    es = np.real(tc.backend.numpy(es))
    return es[0], (es[1::2] - es[2::2]) / 2


e, g = energy_and_gradient(theta0)
print("exact:", round(e, 4), np.round(g, 4))

Now on the hardware. Each step is one call, i.e. one batch submission; all tasks go to the task store.

In [ ]:
lr, steps = 0.2, 3
theta, exact_theta = theta0.copy(), theta0.copy()
for step in range(steps):
    e, g = energy_and_gradient(theta, device=d)
    e_exact, g_exact = energy_and_gradient(exact_theta)
    print(
        f"step {step}: energy {e:.4f} (exact path {e_exact:.4f}), gradient {np.round(g, 3)}"
    )
    theta, exact_theta = theta - lr * g, exact_theta - lr * g_exact
e = batch_expectation_ps(
    compiled, pss, device=d, ws=ws, shots=1000, params=bind(theta), info=info
)
print(f"step {steps}: energy {np.real(e):.4f}")

Comparing with the exact gradient descent path computed alongside by simulation shows how far shot noise and gate errors move the hardware energies and gradients. Readout errors are already corrected by the device calibration; for gate errors, see the error mitigation methods of `tc.results.qem`, which `batch_expectation_ps` accepts as `variants`.

The circuits of the whole experiment can be found in the task store afterwards:

In [ ]:
rows = store.find(provider="lqcloud", since=record["created_at"])
batches = {r["batch_id"] for r in rows if r["batch_id"]}
print(len(rows), "circuits in this tutorial,", len(batches), "batch submissions")

## Summary

- `list_properties` gives the calibration of a device, `compile_circuit` and `DefaultCompiler(properties=...)` place circuits on its best qubits.
- `apis.set_store(True)` records every submission from intent to results; paid submissions are sent once, and uncertain ones are recovered by `store.bind`.
- `batch_expectation_ps` turns Pauli string expectations into grouped, batched and readout-corrected hardware runs, with the same call simulating exactly when `device=None`.
- A `SymbolCircuit` compiled once is bound by `params` in one batch per optimization step.

A script version of the parameter-shift VQE is [examples/cloud_parameter_shift_vqe.py](https://github.com/tensorcircuit/tensorcircuit-ng/blob/master/examples/cloud_parameter_shift_vqe.py). To add a new cloud provider, see the [provider guide](../cloud_provider.md).